In [13]:
import os

folder_path = r"C:\Users\Admin\Downloads\supply chain"

for file in os.listdir(folder_path):
    print(file)

!pip install pandas numpy pulp openpyxl


# ============================================================
# MODULE 3: PRESCRIPTIVE OPTIMIZATION ENGINE
# Supply Chain Optimization using PuLP
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
from pulp import (
    LpProblem,
    LpMaximize,
    LpVariable,
    LpStatus,
    lpSum,
    value,
    PULP_CBC_CMD
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

# CHANGE ONLY THIS PATH IF YOUR CSV IS IN A DIFFERENT LOCATION
FILE_PATH = r"C:\Users\Admin\Downloads\supply chain\supply_chain_feature_engineered.csv"

# Output files
OUTPUT_CSV = "module3_optimization_results.csv"
OUTPUT_EXCEL = "module3_optimization_results.xlsx"

# Business constraints
TOTAL_BUDGET = 500000
TOTAL_CAPACITY = 10000
MAX_INVESTMENT_PER_PRODUCT = 3000
MIN_INVESTMENT_PER_PRODUCT = 0

# Number of best alternatives to display
TOP_N = 10


# ============================================================
# 2. LOAD DATA
# ============================================================

print("=" * 70)
print("MODULE 3 - PRESCRIPTIVE OPTIMIZATION ENGINE")
print("=" * 70)

if not os.path.exists(FILE_PATH):
    print("\nERROR: CSV file was not found.")
    print("\nCurrent path:")
    print(FILE_PATH)
    print("\nPlease check the FILE_PATH at the top of the program.")
    raise FileNotFoundError(FILE_PATH)

print("\n[1] Loading dataset...")

df = pd.read_csv(FILE_PATH)

print("Dataset loaded successfully.")
print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nAvailable columns:")
for column in df.columns:
    print(" -", column)


# ============================================================
# 4. AUTOMATIC COLUMN DETECTION
# ============================================================

def find_column(dataframe, possible_names):
    """
    Find a column using a list of possible column names.
    """
    for name in possible_names:
        if name in dataframe.columns:
            return name

    # Partial matching
    for column in dataframe.columns:
        for name in possible_names:
            if name in column:
                return column

    return None


product_col = find_column(
    df,
    [
        "product",
        "product_name",
        "sku",
        "sku_name",
        "item",
        "item_name",
        "product_type"
    ]
)

cost_col = find_column(
    df,
    [
        "cost",
        "unit_cost",
        "purchase_cost",
        "procurement_cost",
        "price",
        "unit_price"
    ]
)

price_col = find_column(
    df,
    [
        "selling_price",
        "sales_price",
        "revenue",
        "unit_revenue",
        "sellingprice"
    ]
)

demand_col = find_column(
    df,
    [
        "demand",
        "forecast_demand",
        "predicted_demand",
        "expected_demand",
        "demand_forecast"
    ]
)

capacity_col = find_column(
    df,
    [
        "capacity",
        "production_capacity",
        "available_capacity",
        "warehouse_capacity"
    ]
)


# ============================================================
# 5. VALIDATE REQUIRED COLUMNS
# ============================================================

print("\n[2] Detecting required columns...")

print("Product column :", product_col)
print("Cost column    :", cost_col)
print("Price column   :", price_col)
print("Demand column  :", demand_col)
print("Capacity       :", capacity_col)


# ============================================================
# 6. CREATE FALLBACK COLUMNS IF NECESSARY
# ============================================================

# Product
if product_col is None:
    df["product"] = ["Product_" + str(i + 1) for i in range(len(df))]
    product_col = "product"


# Cost
if cost_col is None:

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    if len(numeric_columns) > 0:
        cost_col = numeric_columns[0]
        print(
            "\nWARNING: Cost column was not explicitly identified."
        )
        print(
            "Using numeric column as cost:",
            cost_col
        )
    else:
        df["cost"] = 100
        cost_col = "cost"


# Selling price
if price_col is None:

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    if len(numeric_columns) > 1:
        price_col = numeric_columns[1]
        print(
            "\nWARNING: Selling price column was not explicitly identified."
        )
        print(
            "Using numeric column as selling price:",
            price_col
        )
    else:
        df["selling_price"] = df[cost_col] * 1.25
        price_col = "selling_price"


# Demand
if demand_col is None:

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    if len(numeric_columns) > 2:
        demand_col = numeric_columns[2]

        print(
            "\nWARNING: Demand column was not explicitly identified."
        )

        print(
            "Using numeric column as demand:",
            demand_col
        )

    else:
        df["demand"] = 100
        demand_col = "demand"


# Capacity
if capacity_col is None:
    df["capacity"] = TOTAL_CAPACITY
    capacity_col = "capacity"


# ============================================================
# 7. CONVERT NUMERIC COLUMNS
# ============================================================

numeric_required = [
    cost_col,
    price_col,
    demand_col,
    capacity_col
]

for column in numeric_required:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ============================================================
# 8. REMOVE INVALID DATA
# ============================================================

df = df.dropna(
    subset=[
        cost_col,
        price_col,
        demand_col
    ]
).copy()

df = df[
    (df[cost_col] >= 0) &
    (df[price_col] >= 0) &
    (df[demand_col] >= 0)
].copy()

df = df.reset_index(drop=True)


# ============================================================
# 9. CREATE OPTIMIZATION DATA
# ============================================================

df["unit_cost"] = df[cost_col]

df["unit_price"] = df[price_col]

df["forecast_demand"] = df[demand_col]

df["unit_profit"] = (
    df["unit_price"] -
    df["unit_cost"]
)

df["maximum_quantity"] = np.minimum(
    df["forecast_demand"],
    MAX_INVESTMENT_PER_PRODUCT
)

df["maximum_quantity"] = (
    df["maximum_quantity"]
    .fillna(0)
    .clip(lower=0)
)


# ============================================================
# 10. CREATE OPTIMIZATION MODEL
# ============================================================

print("\n[3] Creating optimization model...")

model = LpProblem(
    "Supply_Chain_Prescriptive_Optimization",
    LpMaximize
)


# ============================================================
# 11. DECISION VARIABLES
# ============================================================

decision_variables = {}

for i in df.index:

    max_quantity = int(
        np.floor(
            df.loc[i, "maximum_quantity"]
        )
    )

    decision_variables[i] = LpVariable(
        f"quantity_{i}",
        lowBound=MIN_INVESTMENT_PER_PRODUCT,
        upBound=max_quantity,
        cat="Integer"
    )


# ============================================================
# 12. OBJECTIVE FUNCTION
# ============================================================

model += lpSum(
    decision_variables[i] *
    df.loc[i, "unit_profit"]
    for i in df.index
)


# ============================================================
# 13. BUSINESS CONSTRAINT 1: TOTAL BUDGET
# ============================================================

model += (
    lpSum(
        decision_variables[i] *
        df.loc[i, "unit_cost"]
        for i in df.index
    )
    <= TOTAL_BUDGET
), "Total_Budget"


# ============================================================
# 14. BUSINESS CONSTRAINT 2: TOTAL CAPACITY
# ============================================================

model += (
    lpSum(
        decision_variables[i]
        for i in df.index
    )
    <= TOTAL_CAPACITY
), "Total_Capacity"


# ============================================================
# 15. BUSINESS CONSTRAINT 3: DEMAND LIMIT
# ============================================================

for i in df.index:

    model += (
        decision_variables[i]
        <= df.loc[i, "forecast_demand"]
    ), f"Demand_Limit_{i}"


# ============================================================
# 16. SOLVE MODEL
# ============================================================

print("\n[4] Solving optimization problem...")

solver = PULP_CBC_CMD(
    msg=False
)

model.solve(solver)


# ============================================================
# 17. CHECK SOLUTION
# ============================================================

status = LpStatus[
    model.status
]

print("\nOptimization status:", status)

if status != "Optimal":

    print(
        "\nERROR: An optimal solution could not be found."
    )

    print(
        "Solver status:",
        status
    )

    raise RuntimeError(
        "Optimization did not return an optimal solution."
    )


# ============================================================
# 18. EXTRACT OPTIMAL SOLUTION
# ============================================================

df["optimal_quantity"] = [
    decision_variables[i].value()
    for i in df.index
]

df["optimal_quantity"] = (
    df["optimal_quantity"]
    .fillna(0)
    .round(0)
    .astype(int)
)


# ============================================================
# 19. CALCULATE FINANCIAL METRICS
# ============================================================

df["investment"] = (
    df["optimal_quantity"] *
    df["unit_cost"]
)

df["revenue"] = (
    df["optimal_quantity"] *
    df["unit_price"]
)

df["profit"] = (
    df["optimal_quantity"] *
    df["unit_profit"]
)

df["profit_margin"] = np.where(
    df["revenue"] > 0,
    (
        df["profit"] /
        df["revenue"]
    ) * 100,
    0
)


# ============================================================
# 20. CALCULATE UTILIZATION
# ============================================================

df["demand_utilization"] = np.where(
    df["forecast_demand"] > 0,
    (
        df["optimal_quantity"] /
        df["forecast_demand"]
    ) * 100,
    0
)


# ============================================================
# 21. TOTAL RESULTS
# ============================================================

total_quantity = df[
    "optimal_quantity"
].sum()

total_investment = df[
    "investment"
].sum()

total_revenue = df[
    "revenue"
].sum()

total_profit = df[
    "profit"
].sum()


# ============================================================
# 22. DISPLAY OPTIMAL SOLUTION
# ============================================================

print("\n" + "=" * 70)
print("OPTIMAL SUPPLY CHAIN DECISION")
print("=" * 70)

print(
    f"\nTotal Quantity     : {total_quantity:,.0f}"
)

print(
    f"Total Investment   : {total_investment:,.2f}"
)

print(
    f"Total Revenue      : {total_revenue:,.2f}"
)

print(
    f"Maximum Profit     : {total_profit:,.2f}"
)


# ============================================================
# 23. RANK PRODUCTS
# ============================================================

ranking = df[
    [
        product_col,
        "unit_cost",
        "unit_price",
        "unit_profit",
        "forecast_demand",
        "optimal_quantity",
        "investment",
        "revenue",
        "profit",
        "profit_margin",
        "demand_utilization"
    ]
].copy()

ranking = ranking.sort_values(
    by="profit",
    ascending=False
).reset_index(drop=True)

ranking["rank"] = (
    ranking["profit"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)

ranking = ranking[
    [
        "rank",
        product_col,
        "unit_cost",
        "unit_price",
        "unit_profit",
        "forecast_demand",
        "optimal_quantity",
        "investment",
        "revenue",
        "profit",
        "profit_margin",
        "demand_utilization"
    ]
]


# ============================================================
# 24. DISPLAY TOP ALTERNATIVES
# ============================================================

print("\n" + "=" * 70)
print("TOP RANKED ALTERNATIVES")
print("=" * 70)

print(
    ranking.head(TOP_N).to_string(
        index=False
    )
)


# ============================================================
# 25. CREATE ALTERNATIVE SCENARIOS
# ============================================================

print("\n" + "=" * 70)
print("GENERATING ALTERNATIVE ACTIONS")
print("=" * 70)


def solve_scenario(
    budget,
    capacity,
    max_per_product
):

    scenario = LpProblem(
        "Alternative_Action",
        LpMaximize
    )

    variables = {}

    for i in df.index:

        maximum = int(
            min(
                df.loc[i, "forecast_demand"],
                max_per_product
            )
        )

        variables[i] = LpVariable(
            f"x_{i}",
            lowBound=0,
            upBound=maximum,
            cat="Integer"
        )

    scenario += lpSum(
        variables[i] *
        df.loc[i, "unit_profit"]
        for i in df.index
    )

    scenario += lpSum(
        variables[i] *
        df.loc[i, "unit_cost"]
        for i in df.index
    ) <= budget

    scenario += lpSum(
        variables[i]
        for i in df.index
    ) <= capacity

    for i in df.index:

        scenario += (
            variables[i]
            <= df.loc[i, "forecast_demand"]
        )

    scenario.solve(
        PULP_CBC_CMD(msg=False)
    )

    if LpStatus[
        scenario.status
    ] != "Optimal":

        return None

    quantity = sum(
        variables[i].value()
        for i in df.index
    )

    investment = sum(
        variables[i].value() *
        df.loc[i, "unit_cost"]
        for i in df.index
    )

    revenue = sum(
        variables[i].value() *
        df.loc[i, "unit_price"]
        for i in df.index
    )

    profit = sum(
        variables[i].value() *
        df.loc[i, "unit_profit"]
        for i in df.index
    )

    return {
        "Total_Quantity": quantity,
        "Investment": investment,
        "Revenue": revenue,
        "Profit": profit
    }


# ============================================================
# 26. GENERATE BUSINESS SCENARIOS
# ============================================================

scenario_definitions = [

    (
        "Conservative",
        TOTAL_BUDGET * 0.60,
        TOTAL_CAPACITY * 0.60,
        MAX_INVESTMENT_PER_PRODUCT * 0.60
    ),

    (
        "Balanced",
        TOTAL_BUDGET * 0.80,
        TOTAL_CAPACITY * 0.80,
        MAX_INVESTMENT_PER_PRODUCT * 0.80
    ),

    (
        "Recommended",
        TOTAL_BUDGET,
        TOTAL_CAPACITY,
        MAX_INVESTMENT_PER_PRODUCT
    ),

    (
        "Aggressive",
        TOTAL_BUDGET * 1.20,
        TOTAL_CAPACITY * 1.20,
        MAX_INVESTMENT_PER_PRODUCT * 1.20
    )
]


scenario_results = []


for (
    name,
    budget,
    capacity,
    max_product
) in scenario_definitions:

    result = solve_scenario(
        budget,
        capacity,
        max_product
    )

    if result is not None:

        result["Scenario"] = name

        result["Budget"] = budget

        result["Capacity"] = capacity

        scenario_results.append(
            result
        )


# ============================================================
# 27. RANK SCENARIOS
# ============================================================

scenario_df = pd.DataFrame(
    scenario_results
)

if not scenario_df.empty:

    scenario_df = scenario_df[
        [
            "Scenario",
            "Budget",
            "Capacity",
            "Total_Quantity",
            "Investment",
            "Revenue",
            "Profit"
        ]
    ]

    scenario_df = scenario_df.sort_values(
        by="Profit",
        ascending=False
    ).reset_index(drop=True)

    scenario_df["Rank"] = (
        scenario_df["Profit"]
        .rank(
            method="dense",
            ascending=False
        )
        .astype(int)
    )

    scenario_df = scenario_df[
        [
            "Rank",
            "Scenario",
            "Budget",
            "Capacity",
            "Total_Quantity",
            "Investment",
            "Revenue",
            "Profit"
        ]
    ]


# ============================================================
# 28. DISPLAY SCENARIO RANKING
# ============================================================

print("\n" + "=" * 70)
print("SCENARIO RANKING")
print("=" * 70)

if not scenario_df.empty:

    print(
        scenario_df.to_string(
            index=False
        )
    )


# ============================================================
# 29. SAVE PRODUCT RESULTS
# ============================================================

ranking.to_csv(
    OUTPUT_CSV,
    index=False
)


# ============================================================
# 30. SAVE COMPLETE EXCEL REPORT
# ============================================================

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl"
) as writer:

    ranking.to_excel(
        writer,
        sheet_name="Product Ranking",
        index=False
    )

    scenario_df.to_excel(
        writer,
        sheet_name="Scenario Ranking",
        index=False
    )

    df.to_excel(
        writer,
        sheet_name="Optimization Details",
        index=False
    )


# ============================================================
# 31. FINAL RECOMMENDATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL PRESCRIPTIVE RECOMMENDATION")
print("=" * 70)

if not scenario_df.empty:

    best_scenario = scenario_df.iloc[0]

    print(
        "\nRecommended Action :",
        best_scenario["Scenario"]
    )

    print(
        "Expected Investment :",
        f"{best_scenario['Investment']:,.2f}"
    )

    print(
        "Expected Revenue    :",
        f"{best_scenario['Revenue']:,.2f}"
    )

    print(
        "Expected Profit     :",
        f"{best_scenario['Profit']:,.2f}"
    )

    print(
        "Expected Quantity   :",
        f"{best_scenario['Total_Quantity']:,.0f}"
    )


# ============================================================
# 32. FILE OUTPUT CONFIRMATION
# ============================================================

print("\n" + "=" * 70)
print("FILES GENERATED SUCCESSFULLY")
print("=" * 70)

print(
    "\nCSV file   :",
    os.path.abspath(OUTPUT_CSV)
)

print(
    "Excel file :",
    os.path.abspath(OUTPUT_EXCEL)
)

print("\nModule 3 execution completed successfully.")

DataCoSupplyChainDataset.csv
delay_y_test.csv
delay_y_train.csv
DescriptionDataCoSupplyChain.csv
supply_chain_clean.csv
supply_chain_feature_engineered.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


MODULE 3 - PRESCRIPTIVE OPTIMIZATION ENGINE

[1] Loading dataset...
Dataset loaded successfully.
Rows    : 180519
Columns : 56

Available columns:
 - type
 - days_for_shipping_real
 - days_for_shipment_scheduled
 - benefit_per_order
 - sales_per_customer
 - delivery_status
 - late_delivery_risk
 - category_id
 - category_name
 - customer_city
 - customer_country
 - customer_id
 - customer_segment
 - customer_state
 - department_id
 - department_name
 - latitude
 - longitude
 - market
 - order_city
 - order_country
 - order_customer_id
 - order_date_dateorders
 - order_id
 - order_item_cardprod_id
 - order_item_discount
 - order_item_discount_rate
 - order_item_id
 - order_item_product_price
 - order_item_profit_ratio
 - order_item_quantity
 - sales
 - order_item_total
 - order_profit_per_order
 - order_region
 - order_state
 - order_status
 - product_card_id
 - product_category_id
 - product_image
 - product_name
 - product_price
 - product_status
 - shipping_date_dateorders
 - shippin